# Colab-X-Local-Model server
Run every cell top to bottom. The last cell blocks and prints a `Public tunnel URL` line — copy that URL into `ui/index.html`'s `BASE_URL` constant.

No signup or auth token needed — the tunnel uses Cloudflare's free `cloudflared` quick tunnel. The first run downloads the `cloudflared` binary automatically.

This notebook runs a GGUF model via `llama-cpp-python`. To use a different GGUF model, change `HF_REPO_ID` and `GGUF_FILENAME` in the cell below before running it — nothing else needs to change.

The `llama-cpp-python` install cell compiles with CUDA support, which takes a few minutes on first run.

In [ ]:
!pip install -q flask==3.0.3 flask-cors==4.0.1 huggingface_hub
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --no-cache-dir

In [ ]:
# The one place to change the model — pick per experiment.
# This repo is GGUF-only (no plain HF checkpoint), so llama-cpp-python loads
# a specific quantized .gguf file from it rather than a transformers pipeline.
HF_REPO_ID = "JonathanColetti/Qwen3.8-27B-Uncensored-GGUF"
GGUF_FILENAME = "Qwen3.8-27B-Uncensored-noMTP-IQ2_M.gguf"  # ~10.2GB, fits a free-tier T4 (16GB VRAM)

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=GGUF_FILENAME)

llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=8192, verbose=False)

# Sanity check inside the notebook before wiring up Flask.
print(llm("Hello, my name is", max_tokens=20, echo=False)["choices"][0]["text"])

In [ ]:
from flask import Flask, jsonify, request
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


@app.post("/generate")
def generate():
    data = request.get_json(silent=True) or {}
    prompt = data.get("prompt", "")
    if not isinstance(prompt, str) or not prompt.strip():
        return jsonify({"error": "prompt is required"}), 400

    try:
        max_new_tokens = min(int(data.get("max_new_tokens", 200) or 200), 512)
    except (TypeError, ValueError):
        max_new_tokens = 200

    try:
        result = llm(prompt, max_tokens=max_new_tokens, echo=False)
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500

    text = result["choices"][0]["text"]
    return jsonify({"response": text})

In [ ]:
# This cell blocks — the tunnel URL is printed above the running server log.
import subprocess
import threading
import re


def run_flask():
    app.run(host="0.0.0.0", port=5000)


threading.Thread(target=run_flask, daemon=True).start()

# Install cloudflared if not already present (Colab has no cloudflared preinstalled).
import os

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True,
    )

process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for line in process.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        print(f"\nPublic tunnel URL: {match.group(0)}")
        print("Copy this into ui/index.html's BASE_URL constant.\n")